Ingest Dimension Data into Bronze

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType,FloatType

In [0]:
%run /Workspace/Users/yklynk@gmail.com/Azure_databricks_data_engineering_project_shopvista_ecomm/1_setup/utilities

In [0]:
print (bronze_schema, silver_schema, gold_schema)

In [0]:
#  Define Widget Parameters for Shopvista Source Tables
dbutils.widgets.text('catalog','shopvista', 'catalog')
dbutils.widgets.text('brands_source', 'brands', 'brands_source' )
dbutils.widgets.text('category_source', 'category', 'category_source')
dbutils.widgets.text('customers_source', 'customers', 'customers_source' )
dbutils.widgets.text('date_source', 'date', 'date_source')
dbutils.widgets.text('product_source', 'products', 'product_source')

## BRANDS

In [0]:
# Retrieve and Print Catalog & Brands Source Widget Values
catalog = dbutils.widgets.get('catalog')
brands_source = dbutils.widgets.get('brands_source')

print(catalog, brands_source)

In [0]:
# Define schema for the brands data

brands_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
    
])

In [0]:
# Show the path to the source
raw_data_path = f"/Volumes/{catalog}/raw/raw_landing/{brands_source}/*.csv"

In [0]:
#Read data into dataframe and add metadata

df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferschema", True)
    .load(raw_data_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_path")
)



In [0]:
# write data to bronze table
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{brands_source}")

## CATEGORY

In [0]:
# Retrieve and Print Catalog & Category Source Widget Values
catalog = dbutils.widgets.get('catalog')
category_source = dbutils.widgets.get('category_source')
print(catalog, category_source)

In [0]:
# Define schema for the category data
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

In [0]:
# Show the path to the source
raw_data_path = f"/Volumes/{catalog}/raw/raw_landing/{category_source}/*.csv"

In [0]:
#load data into dataframe and add metadata

df = (spark.read.format("csv")
            .option("header", True)
            .schema(category_schema)
            .load(raw_data_path)
            .withColumn("read_timestamp", F.current_timestamp()) #add metadata colum
            .select("*", "_metadata.file_name", "_metadata.file_path") #add metadata colum
)



In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{category_source}")

In [0]:
display(df.limit(2))

**CUSTOMERS**

In [0]:
# Retrieve and Print Catalog & Customers Source Widget Values
catalog = dbutils.widgets.get('catalog')
customers_source = dbutils.widgets.get('customers_source')
print(catalog, customers_source)


In [0]:
# Define schema for the customer data

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

In [0]:
# Show the path to the source
raw_data_path = f"/Volumes/{catalog}/raw/raw_landing/{customers_source}/*.csv"

In [0]:
#load data into dataframe and add metadata

df = (
    spark.read.format("csv")
        .option("header",True)
        .load(raw_data_path)
        .withColumn("read_timestamp", F.current_timestamp()) # add metadata column
        .select("*", "_metadata.file_name", "_metadata.file_path") #add metadata column

)


In [0]:
# write customer to bronze table

df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{customers_source}")    

In [0]:
display(df.limit(5))

## DATE

In [0]:
# Retrieve and Print Catalog & Date Source Widget Values
catalog = dbutils.widgets.get("catalog")
date_source = dbutils.widgets.get("date_source")

print(catalog, date_source)

In [0]:
# Define schema for the date data

date_schema = StructType([
    StructField("date", DateType(), True),
    StructField("year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True)
    

])

In [0]:
# Show the path to the source
raw_data_path = f"/Volumes/{catalog}/raw/raw_landing/{date_source}/*.csv"

In [0]:

#load data into dataframe and add metadata
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(date_schema)
        .load(raw_data_path)
        .withColumn("read_timestamp", F.current_timestamp()) # Add metadata columns
        .select("*", "_metadata.file_name", "_metadata.file_path") # Add metadata columns
)



In [0]:

# write date to bronze table
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .option("overwriteSchema", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{date_source}")

In [0]:
display(df.limit(5))

## PRODUCTS

In [0]:
# Retrieve and Print Catalog & Products Source Widget Values
catalog = dbutils.widgets.get("catalog")
product_source = dbutils.widgets.get("product_source")

print(catalog, product_source)

In [0]:
# Define schema for product table

product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("Sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True), #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True), #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True)
])

In [0]:
# Show the path to the source
raw_data_path = f"/Volumes/{catalog}/raw/raw_landing/products/*.csv"

In [0]:

# load data into dataframe and add metadata
df = (
    spark.read.format("csv")
        .option("header", True)
        .schema(product_schema)
        .load(raw_data_path)
        .withColumn("data_source", F.current_timestamp()) # add metadata columns
        .select("*", "_metadata.file_name", "_metadata.file_path") # add metadata columns
)

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.products")

In [0]:
display(df.limit(5))